In [61]:
%pip install oracledb
%pip install pandas

4063.25s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


4069.02s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [62]:
import oracledb
import pandas 

# Oracle 연결 정보
# DSN = 호스트 + 포트 + 서비스명 (SID)
dsn = "oracle-db/XEPDB1"

try:
    # oracledb.connect() 함수로 데이터베이스 연결
    conn = oracledb.connect(
        user="system",
        password="oracle",
        dsn=dsn
    )
    print("Oracle 서버에 성공적으로 연결되었습니다!")
except oracledb.Error as err:
    print("Oracle 연결 실패:", err)

Oracle 서버에 성공적으로 연결되었습니다!


In [64]:
# 커서 생성
cursor = conn.cursor()
print("Oracle SQL 커서 준비 완료")

Oracle SQL 커서 준비 완료


In [65]:
# 데이터베이스 생성
query = """
    CREATE DATABASE IF NOT EXISTS pydb4
"""

conn.close()
print("데이터베이스 생성 완료")

데이터베이스 생성 완료


In [66]:
import oracledb

try:
    conn = oracledb.connect(
        user="system",
        password="oracle",
        dsn="oracle-db/XEPDB1"
    )
    print("Oracle 서버 (pydb4 데이터베이스) 연결 성공!")
except oracledb.Error as err:
    print("Oracle 연결 실패:", err)

Oracle 서버 (pydb4 데이터베이스) 연결 성공!


In [67]:
cursor = conn.cursor()
print("Oracle SQL 쿼리 준비")

Oracle SQL 쿼리 준비


In [68]:
# Oracle 테이블 생성문 (NOTES: Oracle용 자동증가 및 PK 문법)
# Oracle: NUMBER, VARCHAR2, DEFAULT, NOT NULL, PRIMARY KEY
query = """
    CREATE TABLE IF NOT EXISTS users (
        id NUMBER(10) GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        name VARCHAR2(50) NOT NULL,
        age NUMBER NOT NULL DEFAULT 0,
        city VARCHAR2(50) NOT NULL
    )
"""

print("테이블 생성 완료")

테이블 생성 완료


In [69]:
# 테이블 목록 확인
cursor.execute("SELECT TABLE_NAME FROM USER_TABLES")
tbl = cursor.fetchall()
print("테이블 목록:", tbl)

테이블 목록: [('LOGMNR_SESSION_EVOLVE$',), ('LOGMNR_GLOBAL$',), ('LOGMNR_PDB_INFO$',), ('LOGMNR_DID$',), ('LOGMNR_UID$',), ('LOGMNRGGC_GTLO',), ('LOGMNRGGC_GTCS',), ('LOGMNRC_DBNAME_UID_MAP',), ('LOGMNR_LOG$',), ('LOGMNR_PROCESSED_LOG$',), ('LOGMNR_SPILL$',), ('LOGMNR_AGE_SPILL$',), ('LOGMNR_RESTART_CKPT_TXINFO$',), ('LOGMNR_ERROR$',), ('LOGMNR_RESTART_CKPT$',), ('LOGMNR_FILTER$',), ('LOGMNR_SESSION_ACTIONS$',), ('LOGMNR_PARAMETER$',), ('LOGMNR_SESSION$',), ('LOGMNR_PROFILE_TABLE_STATS$',), ('LOGMNR_PROFILE_PLSQL_STATS$',), ('REDO_DB',), ('REDO_LOG',), ('ROLLING$CONNECTIONS',), ('ROLLING$DATABASES',), ('ROLLING$DIRECTIVES',), ('ROLLING$EVENTS',), ('ROLLING$PARAMETERS',), ('ROLLING$PLAN',), ('ROLLING$STATISTICS',), ('ROLLING$STATUS',), ('MVIEW$_ADV_WORKLOAD',), ('MVIEW$_ADV_BASETABLE',), ('MVIEW$_ADV_SQLDEPEND',), ('MVIEW$_ADV_PRETTY',), ('MVIEW$_ADV_TEMP',), ('MVIEW$_ADV_FILTER',), ('MVIEW$_ADV_LOG',), ('MVIEW$_ADV_FILTERINSTANCE',), ('MVIEW$_ADV_LEVEL',), ('MVIEW$_ADV_ROLLUP',), ('MVIEW$_A

In [70]:
# 테이블 구조 확인
# Oracle DESCRIBE 는 권한 문제로 실패할 수 있으므로
# USER_TAB_COLUMNS 관리를 통해 직접 확인
query = """
    SELECT COLUMN_NAME, DATA_TYPE, DATA_LENGTH, NULLABLE
    FROM USER_TAB_COLUMNS
    WHERE TABLE_NAME = 'USERS'
    ORDER BY COLUMN_ID
"""
cursor.execute(query)
desc_tbl = cursor.fetchall()
print("테이블 구조:")
for col in desc_tbl:
    print(f"  {col[0]}: {col[1]} (LEN={col[2]}, {col[3].upper()})")

테이블 구조:


In [71]:
# 단일 데이터 삽입
sql = """
    INSERT INTO users (name, age, city)
    VALUES (:name, :age, :city)
"""
values = {
    'name': 'Alice',
    'age': 22,
    'city': 'Seoul'
}
# cursor.execute(sql, values)
conn.commit()
print("1개 데이터 삽입 완료, ID:", cursor.rowcount)

1개 데이터 삽입 완료, ID: 0


In [72]:
# 복수 데이터 삽입
sql = """
    INSERT INTO users (name, age, city)
    VALUES (:name, :age, :city)
"""
values_list = [
    ('Bob', 30, 'Busan'),
    ('Charlie', 28, 'Incheon'),
    ('Diana', 35, 'Daegu'),
    ('Ethan', 26, 'Gwangju'),
    ('Fiona', 40, 'Jeju'),
    ('George', 33, 'Seoul'),
    ('Hana', 27, 'Suwon'),
    ('Ian', 29, 'Ulsan'),
    ('Jisoo', 24, 'Daejeon'),
    ('Kevin', 31, 'Changwon')
]
# cursor.executemany(sql, values_list)
print("여러개 데이터 삽입 완료.")
conn.commit()

여러개 데이터 삽입 완료.


In [73]:
# 데이터 조회
sql = """
    SELECT name, age, city
    FROM users
    WHERE age >= 30
"""

try:
    cursor.execute(sql)
    data = cursor.fetchall()
    if not data:
        print("조회된 데이터가 없습니다.")
    for d in data:
        print(d)
except oracledb.DatabaseError as e:
    error, = e.args
    print(f"오류 발생 코드: {error.code}")
    print(f"오류 메시지: {error.message}")

오류 발생 코드: 942
오류 메시지: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/


In [74]:
# 업데이트
sql = """
    UPDATE users
    SET age = :age
    WHERE name = :name
"""
values = {'age': 31, 'name': 'Bob'}

conn.commit()
print("수정된 행 개수:", cursor.rowcount)

수정된 행 개수: 0


In [75]:
conn.close()
print("Oracle 연결 종료")

Oracle 연결 종료
